## Session 3, in class: value function iteration off the grid

Computational Macro (HWS 2026), University of Mannheim

Last week the household could only choose capital **on** the grid. Today the choice $k'$ is
continuous. For that we need two tools: a way to evaluate $v$ **between** grid points
(interpolation) and a way to maximize a continuous function (golden section search). Every gap is
marked `___` and belongs to one line of the slides, the comment next to it says what goes in. Each part ends with a small test or a number to compare: do not move on before it fits.

**The problem** is the growth model of sessions 1 and 2,

$$
v(k) = \max_{k_1 \leq k' \leq \min\{k^\alpha, k_N\}} \bigl\{ \ln(k^\alpha - k') + \beta \, \hat v(k') \bigr\},
$$

where $\hat v$ interpolates the vector $\mathbf{v}$ between the grid points. The closed form is the
benchmark: $v(k) = E + F \ln k$ and $k' = \alpha\beta k^\alpha$.

**Plan**

| Part | Content | Time |
| --- | --- | --- |
| 1 | Setup, and a closer look at last week's policy | 5 min |
| 2 | Tool 1: linear interpolation, and the cubic spline from a package | 12 min |
| 3 | Tool 2: maximizing a continuous function | 5 min |
| 4 | Value function iteration off the grid | 8 min |
| 5 | At home: Howard's improvement algorithm | |
| 6 | At home: experiments | |


**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $x_i, f_i$ | interpolation nodes and the values there | `xgrid`, `fgrid` |
| $w = \frac{x - x_i}{x_{i+1} - x_i}$ | weight on the right neighbour | `w` |
| $\hat v(k')$ | interpolated value function | `vhat` (a function) |
| $[a, b]$ | bracket of the golden section search | `a`, `b` |
| $b_i = \min\{k_i^\alpha, k_N\}$ | upper end of the bracket at state $k_i$ | `b` |
| $\mathbf{v}$, $\mathbf{v}^{\text{new}}$ | value function on the grid | `v`, `vnew` |
| $k'_i$ | policy at state $k_i$, now a real number and not an index | `kprime` |
| $\boldsymbol{\Gamma}$ | interpolation weights at the policy (part 5) | `Gamma` |

Everything else is named as last week: `par`, `mpar`, `gri.k`, `util`, `v_true`, `kprime_true`, `distVF`, `iterVF`.

### 0. Packages

The same as last week, plus one package for the cubic spline, `scipy`, which is already part of the
`compmacro` environment. Linear interpolation and the maximization are written by hand, so that you
see what a package does inside.

**If the cell below fails** with `ModuleNotFoundError: No module named 'scipy'`, check that the
notebook runs in the `compmacro` kernel (top right corner of the notebook). If it does, open a
terminal in VS Code (Terminal, New Terminal) and run `conda install -n compmacro scipy` (or
`pip install scipy` if you installed without conda), then restart the kernel.

In [ ]:
import time
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline               # new today: the cubic spline comes from a package

### 1. Setup: parameters, grid, utility, closed form

Identical to last week, with two changes in the numerical parameters: the grid has only
$N = 50$ points, and `interp` selects how $v$ is evaluated between grid points.

In [ ]:
@dataclass(frozen=True)
class EconomicParameters:
    alpha: float = 0.3      # capital share
    beta: float = 0.96      # discount factor


@dataclass(frozen=True)
class NumericalParameters:
    nk: int = 50            # number of points on the capital grid, N
    kmin_rel: float = 0.1   # lower grid bound, relative to k*
    kmax_rel: float = 2.0   # upper grid bound, relative to k*
    spacing: str = "linear" # "linear" or "log"
    interp: str = "spline"  # "linear" or "spline": how v is evaluated between grid points
    crit: float = 1e-6      # convergence tolerance, epsilon
    maxiter: int = 5000     # iteration cap, the safety net


@dataclass(frozen=True)
class Grids:
    k: np.ndarray           # the capital grid


par = EconomicParameters()
mpar = NumericalParameters()
print("Economic parameters: ", par)
print("Numerical parameters:", mpar)

The grid, the utility function, and the closed form. The grid is built by a function today,
`make_grid`, because the experiments at the end build several grids. Its two cases are those of
last week.

In [ ]:
def make_grid(par, mpar):
    kstar = (par.alpha * par.beta) ** (1 / (1 - par.alpha))
    kmin, kmax = mpar.kmin_rel * kstar, mpar.kmax_rel * kstar
    # two cases. If spacing is "linear", return hands back the equidistant grid and the function stops here.
    # Only if spacing is not "linear" does the code reach the last line, which returns the log spaced grid
    if mpar.spacing == "linear":
        return Grids(k=np.linspace(kmin, kmax, mpar.nk))                        # equidistant in k
    return Grids(k=np.exp(np.linspace(np.log(kmin), np.log(kmax), mpar.nk)))   # equidistant in ln k


def util(c):
    return np.log(c)


def closed_form(par, k):
    F_star = par.alpha / (1 - par.alpha * par.beta)
    E_star = (np.log(1 - par.alpha * par.beta) + par.beta * F_star * np.log(par.alpha * par.beta)) / (1 - par.beta)
    return E_star + F_star * np.log(k), par.alpha * par.beta * k ** par.alpha   # value and policy


kstar = (par.alpha * par.beta) ** (1 / (1 - par.alpha))
gri = make_grid(par, mpar)
v_true, kprime_true = closed_form(par, gri.k)
print(f"k* = {kstar:.4f}, {mpar.nk} grid points from {gri.k[0]:.4f} to {gri.k[-1]:.4f}")

### 1. Exercise: a closer look at last week's policy

Last week's figure showed the grid policy on top of the closed form, and the two lines looked
identical. The function below is last week's loop in compact form, with the names of last week.
Solve on the grid with $N = 200$ and **zoom in** on the policy around the steady state,
$k \in [0.9\,k^*, 1.1\,k^*]$.

What do you see, and why can it not be otherwise?

In [ ]:
def solve_on_grid(par, k, crit=1e-6, maxiter=5000):
    # last week's loop in compact form: the reward matrix U once, then the row-wise maximum of M until the distance is small
    meshes_k, meshes_kprime = np.meshgrid(k, k, indexing="ij")    # k_i in row i, k'_j in column j
    C = meshes_k ** par.alpha - meshes_kprime                     # consumption for every (state, choice) pair
    U = np.full(C.shape, -np.inf)                                 # -inf marks the infeasible pairs
    U[C > 0] = util(C[C > 0])
    v, g, distVF, iterVF = np.zeros(len(k)), np.zeros(len(k), dtype=int), [np.inf], 0
    while distVF[-1] > crit and iterVF < maxiter:
        M = U + par.beta * v[None, :]                             # M(v) = U + beta 1 v'
        vnew, g = np.max(M, axis=1), np.argmax(M, axis=1)         # maximum along each row, and the column where it sits
        distVF.append(np.max(np.abs(vnew - v)))
        v = vnew
        iterVF += 1
    return v, k[g]                                                # value, and the policy in levels k_{g_i}


k_on = make_grid(par, NumericalParameters(nk=200)).k    # last week's grid: N = 200, everything else at its default
v_on, kprime_on = solve_on_grid(par, k_on)
kprime_cf = closed_form(par, k_on)[1]                    # the closed form policy on the same grid, for comparison
print(f"on the grid, N = 200: largest policy error {np.max(np.abs(kprime_on - kprime_cf)):.2e}")

In [ ]:
# which grid points lie within 10 percent of the steady state? zoom is a vector of True and False, one entry per grid point
zoom = ___                                               # True where k_on lies between 0.9 * kstar and 1.1 * kstar: two comparisons in brackets, joined by &
print(f"{zoom.sum()} of {len(k_on)} grid points lie in the zoom window")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))            # two panels side by side

# left panel: the policy on the whole grid, against the closed form
ax[0].plot(k_on, kprime_on, label="VFI on the grid")
ax[0].plot(k_on, kprime_cf, "k--", label="closed form")
ax[0].set(title="The whole grid", xlabel="capital today k", ylabel="capital tomorrow k'")

# right panel: only the grid points in the zoom window. k_on[zoom] keeps the entries where zoom is True
ax[1].step(k_on[zoom], kprime_on[zoom], where="mid", label="VFI on the grid")   # a step plot shows the policy as it is: constant, then a jump
ax[1].plot(k_on[zoom], kprime_cf[zoom], "k--", label="closed form")
ax[1].set(title="Zoom around k*", xlabel="capital today k")
ax[1].legend()

plt.tight_layout()
plt.show()

**What you should see.** The policy is a staircase. It can only take the $N$ values on the grid,
so it jumps by one grid spacing at a time, and the error can never be smaller than half a grid
spacing. A finer grid makes the steps smaller, at the cost of $N^2$ entries in $\mathbf{M}$.
The alternative: let the household choose **between** the grid points.

### 2. Tool 1: linear interpolation

We know $f_i = f(x_i)$ at the nodes and want $\hat f(x)$ at any $x$. Find the interval
$[x_i, x_{i+1}]$ that contains $x$, and connect the two neighbours by a straight line:

$$
\hat f(x) = (1 - w)\, f_i + w\, f_{i+1}, \qquad w = \frac{x - x_i}{x_{i+1} - x_i} .
$$

Two steps in code: **locate** the left neighbour $i$, then compute the **weight** $w$.

In [ ]:
def lin_interp(xgrid, fgrid, x):
    # step 1, locate: index i of the left neighbour, x_i <= x < x_{i+1}.
    # searchsorted counts the nodes that are <= x, minus 1 gives the index of the last of them.
    # clip keeps i between the first and the second to last node, so that i + 1 always exists
    i = np.clip(np.searchsorted(xgrid, x, side="right") - 1, 0, len(xgrid) - 2)
    # step 2, weigh: w is the distance from the left neighbour as a share of the interval, the weight on the right neighbour
    w = ___                                              # weigh: the distance of x from the left neighbour xgrid[i], divided by the length of the interval
    return ___                                           # the formula of the slides: weight 1 - w on the left value fgrid[i], weight w on the right value


# test: halfway between the nodes 1 and 2 the interpolant is halfway between 10 and 20
lin_interp(np.array([0.0, 1.0, 2.0]), np.array([0.0, 10.0, 20.0]), 1.5)

### 2. Tool 1, continued: the natural cubic spline, from a package

On every interval $[x_i, x_{i+1}]$ the spline is its own cubic polynomial with **four**
coefficients. The slides give the four conditions that pin them down: the piece goes through
$f_i$ and through $f_{i+1}$, and at the nodes its first and its second derivative equal those of
the neighbouring piece. At the two ends of the grid the natural spline sets the second derivative
to zero. Solving this linear system is what the package does for us:

* Python: `CubicSpline(xgrid, fgrid, bc_type="natural")` from `scipy.interpolate`,
* Julia: `CubicSpline(fgrid, xgrid)` from `DataInterpolations`, **values first, then nodes**.

Both return an object that is called like a function. `make_interpolant` hides the difference
between the two methods: whichever method you pick, it returns a function of $x$. The rest of the
notebook only calls `make_interpolant`.

In [ ]:
def make_interpolant(xgrid, fgrid, method):
    # returns a FUNCTION x -> interpolated value, for either method
    if method == "linear":
        return lambda x: lin_interp(xgrid, fgrid, x)     # our own function from above. lambda x: ... is a function of x that remembers xgrid and fgrid
    return CubicSpline(xgrid, fgrid, bc_type="natural")  # the package solves for the coefficients here, once. The result is called like a function

### 2. Exercise: replicate the figure from the slides

Interpolate two functions on $[-3, 3]$ from **9 nodes** only, once linearly and once with the
spline, and compare both with the truth on a fine grid:

* a wave, $f(x) = \sin(2x)$,
* a function with a kink, $f(x) = \max\{x, 0\}$.

Where does the spline win, and where does it do something the true function does not?

In [ ]:
nodes = np.linspace(-3, 3, 9)                            # the 9 points where we know the function
xfine = np.linspace(-3, 3, 400)                          # a fine grid, to see what happens between the nodes


def wave(x):
    return np.sin(2 * x)                                 # smooth everywhere


def kink(x):
    return np.maximum(x, 0.0)                            # a kink at x = 0, as a borrowing limit will cause

In [ ]:
# the wave: build both interpolants from the 9 values at the nodes, and evaluate them on the fine grid
wave_lin = ___                                                     # as in the line below, with the method "linear"
wave_spl = make_interpolant(nodes, wave(nodes), "spline")(xfine)

# the kink: the same two lines
kink_lin = make_interpolant(nodes, kink(nodes), "linear")(xfine)
kink_spl = make_interpolant(nodes, kink(nodes), "spline")(xfine)

# the largest error on the fine grid, for each function and each method
print(f"wave: largest error linear {np.max(np.abs(wave_lin - wave(xfine))):.3f}, spline {np.max(np.abs(wave_spl - wave(xfine))):.3f}")
print(f"kink: largest error linear {np.max(np.abs(kink_lin - kink(xfine))):.3f}, spline {np.max(np.abs(kink_spl - kink(xfine))):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))            # two panels side by side

# left panel: the wave. Dashed the truth, then the two interpolants, black dots the 9 nodes
ax[0].plot(xfine, wave(xfine), "k--", label="f(x)")
ax[0].plot(xfine, wave_lin, label="linear")
ax[0].plot(xfine, wave_spl, label="spline")
ax[0].plot(nodes, wave(nodes), "ko", markersize=4)
ax[0].set(title="A wave, sin(2x)", xlabel="x")
ax[0].legend()

# right panel: the kink, the same four lines
ax[1].plot(xfine, kink(xfine), "k--", label="f(x)")
ax[1].plot(xfine, kink_lin, label="linear")
ax[1].plot(xfine, kink_spl, label="spline")
ax[1].plot(nodes, kink(nodes), "ko", markersize=4)
ax[1].set(title="A kink, max(x, 0)", xlabel="x")

plt.tight_layout()
plt.show()

**What you should see.** On the smooth wave the spline is far closer than the straight lines,
with the same 9 numbers stored. At the kink the spline overshoots and wiggles, because it insists
on a continuous second derivative that the true function does not have. Keep this in mind for
the borrowing limit in the coming weeks. Our value function $E + F \ln k$ is smooth, so today the
spline is the better tool.

### 2. Exercise: what the spline promises at the nodes

The package did not tell us how it computed the spline. Check the conditions of the slides
yourself, on the wave:

1. the spline goes **through** the 9 points,
2. its **first derivative** is continuous, also at the nodes, where one piece hands over to the next,
3. its **second derivative** is continuous as well, and zero at both ends (natural spline).

Derivatives of a `scipy` spline: `spl(x, 1)` is the first derivative at `x`, `spl(x, 2)` the second.
For comparison, the slope of linear interpolation is one constant per interval.

In [ ]:
spl = CubicSpline(nodes, wave(nodes), bc_type="natural")    # the spline of the wave, kept as an object this time, because we need its derivatives

# condition 1: at the nodes the spline returns the data
print(f"largest gap between spline and data at the nodes: {np.max(np.abs(spl(nodes) - wave(nodes))):.1e}")

# conditions 2 and 3: the derivatives of the spline on the fine grid
d1 = ___                                                    # first derivative of spl at every point of xfine: as in the line below, with order 1
d2 = spl(xfine, 2)                                          # second derivative
d1_lin = np.diff(wave(nodes)) / np.diff(nodes)              # slope of linear interpolation: (f_{i+1} - f_i) / (x_{i+1} - x_i), one per interval
print(f"second derivative at the two ends: {d2[0]:.1e} and {d2[-1]:.1e}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))            # three panels: the spline, its slope, its curvature

# left panel: the spline, one colour per piece. Each piece is its own cubic, and neighbouring pieces meet at the nodes
for i in range(len(nodes) - 1):
    x_piece = np.linspace(nodes[i], nodes[i + 1], 50)    # the interval [x_i, x_{i+1}]
    ax[0].plot(x_piece, spl(x_piece), linewidth=2)
ax[0].plot(nodes, wave(nodes), "ko", markersize=4)
ax[0].set(title="The spline, piece by piece", xlabel="x")

# middle panel: the slope. The slope of the spline is continuous, the slope of linear interpolation jumps at every node
ax[1].plot(xfine, d1, label="spline")
ax[1].step(nodes, np.append(d1_lin, d1_lin[-1]), where="post", label="linear")   # constant on each interval, a jump at each node
ax[1].plot(nodes, spl(nodes, 1), "ko", markersize=4)
ax[1].set(title="First derivative", xlabel="x")
ax[1].legend()

# right panel: the curvature. A straight line on each interval, continuous at the nodes, zero at both ends
ax[2].plot(xfine, d2)
ax[2].plot(nodes, spl(nodes, 2), "ko", markersize=4)
ax[2].axhline(0, color="gray", linewidth=0.8)
ax[2].set(title="Second derivative", xlabel="x")

plt.tight_layout()
plt.show()

**What you should see.** The gap at the nodes is zero up to rounding: the spline goes through
every point. On the left, each colour is one cubic, and the pieces hand over at the nodes without a
visible corner. In the middle, the slope of the spline is a smooth curve, while the slope of linear
interpolation jumps at every node: those jumps are the kinks of the straight lines. On the right,
the second derivative is a straight line on each interval (the second derivative of a cubic is
linear), the lines meet at the nodes, and both ends sit at zero. Four conditions per piece, four
coefficients per piece.

### 3. Tool 2: maximizing a continuous function

Last week the maximum was a search over one row of a table. Now the choice is continuous, and the
computer needs (i) a **bracket** $[a, b]$ to search in and (ii) a rule that shrinks the bracket.
Golden section search evaluates the function at two interior points $c < d$ and throws away the
part of the bracket that cannot contain the maximum:

* if $f(c) > f(d)$, the maximum is in $[a, d]$,
* otherwise it is in $[c, b]$.

The interior points sit at the golden ratio $\varphi = (\sqrt 5 - 1)/2 \approx 0.618$, so that one
of them can be reused and every step costs **one** new function evaluation. The bracket shrinks by
the factor $\varphi$ per step. It needs no derivative, only a function with a single peak.

In [ ]:
def golden_max(f, a, b, tol=1e-10):
    phi = (np.sqrt(5) - 1) / 2                            # the golden ratio, about 0.618
    c, d = b - phi * (b - a), a + phi * (b - a)           # two interior points, c < d
    fc, fd = f(c), f(d)
    while b - a > tol:                                    # until the bracket is narrower than tol
        if fc > fd:                                       # maximum in [a, d]: d becomes the new upper end
            b, d, fd = d, c, fc                           # three updates at once: d is the new upper end b, the old c becomes the new d, and its value fc comes along
            c = b - phi * (b - a)                         # the old c is reused as d, only c is new
            fc = f(c)                                     # the one new function evaluation of this step
        else:                                             # maximum in [c, b]: c becomes the new lower end
            a, c, fc = ___                                   # the mirror image of the line b, d, fd = d, c, fc above: c is the new lower end, the old d becomes the new c, its function value comes along
            d = a + phi * (b - a)                         # the old d is reused as c, only d is new
            fd = f(d)
    x = (a + b) / 2
    return x, f(x)                                        # the maximizer and the maximum


# test: ln(x) + ln(1 - x) peaks at x = 0.5
golden_max(lambda x: np.log(x) + np.log(1 - x), 0.0, 1.0)

### 4. Value function iteration off the grid: one Bellman step

The state is still on the grid, the choice is not. Line 4 and line 5 of the algorithm on the slides:

* line 4: interpolate $\mathbf{v}$, which gives the function $\hat v$,
* line 5, for every state $k_i$: resources are $y_i = k_i^\alpha$, the right hand side is a
  **function** of the choice, $k' \mapsto \ln(y_i - k') + \beta\, \hat v(k')$, the bracket is
  $[k_1, b_i]$ with $b_i = \min\{y_i, k_N\}$ (consumption must stay positive, and we do not want to
  evaluate $\hat v$ outside the grid), and golden section search returns the maximizer $k'_i$ and
  the maximum $v^{\text{new}}_i$.

Compare with last week: the matrix $\mathbf{M}$ and the row-wise maximum are gone, a loop over
states with one search each has taken their place.

In [ ]:
def vfi_update(v, par, mpar, gri):
    vhat = make_interpolant(gri.k, v, mpar.interp)       # line 4: v between the grid points, built once per step
    vnew, kprime = np.empty(mpar.nk), np.empty(mpar.nk)
    for i in range(mpar.nk):                             # line 5: loop over the states k_i
        y = gri.k[i] ** par.alpha                        # resources at k_i
        rhs = lambda kp: ___                             # right hand side as a function of the choice kp: utility of consumption y - kp, plus beta times vhat at kp
        b = min(___, ___)                                # upper end b_i of the bracket: the smaller of resources (minus 1e-10, so that c > 0) and the last grid point
        kprime[i], vnew[i] = ___                            # golden_max(function, lower end, upper end): search rhs between the first grid point and b
    return vnew, kprime

### 4. The loop

Unchanged from last week, except that one line calls `vfi_update` where the matrix $\mathbf{M}$
used to be. The theory is unchanged as well: the presentation today showed that the Bellman operator
is a contraction, so the distance still falls by the factor $\beta$ per iteration.

In [ ]:
timer = time.perf_counter()
v = np.zeros(mpar.nk)                                    # line 2: the guess v_0 = 0
distVF, iterVF = [np.inf], 0                             # the distances d_n, one per iteration, and the counter
while distVF[-1] > mpar.crit and iterVF < mpar.maxiter:  # line 3
    vnew, kprime = ___                                   # lines 4 and 5: one Bellman step, off the grid. Call vfi_update with the current guess v
    dd = np.max(np.abs(vnew - v))                        # line 6: the distance ||v_new - v||_inf
    v = vnew                                             #         and the update
    iterVF += 1
    distVF.append(dd)
time_vfi = time.perf_counter() - timer
if distVF[-1] > mpar.crit:
    raise RuntimeError("not converged: the iteration cap was hit")
print(f"iterations: {iterVF}, final distance: {distVF[-1]:.3e}, last ratio: {distVF[-1] / distVF[-2]:.4f}, time: {time_vfi:.1f} s")

### 4. Checks against the closed form

The same checks as last week. Compare with the grid solution of part 1, which used four times as
many grid points.

In [ ]:
print(f"policy at the grid bounds: {np.sum(kprime <= gri.k[0] + 1e-8)} states at k_1, {np.sum(kprime >= gri.k[-1] - 1e-8)} states at k_N")
print(f"largest policy error off the grid, N = {mpar.nk} ({mpar.interp}): {np.max(np.abs(kprime - kprime_true)):.2e}")
print(f"largest policy error on the grid,  N = 200:          {np.max(np.abs(kprime_on - kprime_cf)):.2e}")
print(f"largest value error: {np.max(np.abs(v - v_true)):.2e}   (certificate beta/(1-beta) d: {par.beta / (1 - par.beta) * distVF[-1]:.2e})")

The zoom of part 1 again, now with the policy off the grid. To see the policy between the states
as well, it is interpolated like any other function on the grid.

In [ ]:
kfine = np.linspace(0.9 * kstar, 1.1 * kstar, 300)       # a fine grid in the zoom window
kprime_fine = make_interpolant(gri.k, kprime, mpar.interp)(kfine)   # the policy off the grid, interpolated between the states

plt.figure(figsize=(6.5, 4))
plt.step(k_on[zoom], kprime_on[zoom], where="mid", label="on the grid, N = 200")
plt.plot(kfine, kprime_fine, label=f"off the grid, N = {mpar.nk}")
plt.plot(kfine, closed_form(par, kfine)[1], "k--", label="closed form")
plt.title("Policy, zoom around k*")
plt.xlabel("capital today k")
plt.ylabel("capital tomorrow k'")
plt.legend()
plt.tight_layout()
plt.show()

**What you should find.** 337 iterations as last week, since the contraction rate is $\beta$
whatever the method. With 50 grid points and the spline the policy error is about $10^{-6}$, a
thousand times smaller than on the grid with 200 points. The value error is $2 \cdot 10^{-5}$ and
no longer comes from the grid: it is the stopping error, bounded by the certificate. The price is
the run time, 337 iterations times 50 searches times about 50 function evaluations. Part 5, at
home, removes most of the iterations.

### 5. At home: Howard's improvement algorithm

Not covered in class. Work through it before next week, the session on policy function iteration
builds on it.

A Bellman step uses the new policy for **one** period and then continues with the old value.
Howard's idea: use the new policy **forever**. The value of following the policy $k'_i$ forever
solves

$$
v_i = u(k_i^\alpha - k'_i) + \beta\, \hat v(k'_i) \qquad \text{for all } i .
$$

Interpolation is **linear in the values** $\mathbf{v}$: $\hat v(k'_i) = \sum_j \Gamma_{ij} v_j$, where row
$i$ of $\boldsymbol{\Gamma}$ holds the interpolation weights at $k'_i$. With linear interpolation the row has two
entries, $1 - w$ on the left neighbour and $w$ on the right one, a lottery over the two
neighbours. The system is therefore linear and can be solved in one step,

$$
\mathbf{v} = \mathbf{u} + \beta \boldsymbol{\Gamma} \mathbf{v}
\quad \Longleftrightarrow \quad
\mathbf{v} = (\mathbf{I} - \beta \boldsymbol{\Gamma})^{-1} \mathbf{u} .
$$

Column $j$ of $\boldsymbol{\Gamma}$ is what the interpolant returns at the policy when $\mathbf{v}$ is the $j$-th unit
vector. That builds $\boldsymbol{\Gamma}$ for the linear interpolant and the spline alike. With the spline the rows
still sum to one, but they have more than two entries and some are negative.

In [ ]:
def howard_step(kprime, par, mpar, gri):
    eye = np.eye(mpar.nk)                                # the identity matrix I: column j is the j-th unit vector, 1 at grid point j and 0 elsewhere
    # column j of Gamma: pretend v is the j-th unit vector, interpolate it, and evaluate it at the policy.
    # The result is the weight that v_j receives in vhat(k'_i), for every state i
    Gamma = np.column_stack([___ for j in range(mpar.nk)])   # as in part 2: an interpolant with nodes gri.k and values eye[:, j], evaluated at kprime
    u = ___                                              # payoff under the policy, one number per state: utility of output minus kprime
    return ___                                           # solve (I - beta Gamma) v = u for v with np.linalg.solve(matrix, right hand side)

The loop: find the policy with one Bellman step, then replace the value by the value of following
that policy forever. Stop when the value no longer moves.

In [ ]:
timer = time.perf_counter()
v_pfi = np.zeros(mpar.nk)
distPF, iterPF = [np.inf], 0
while distPF[-1] > mpar.crit and iterPF < mpar.maxiter:
    _, kprime_pfi = vfi_update(v_pfi, par, mpar, gri)    # policy improvement: the maximization
    vnew = howard_step(kprime_pfi, par, mpar, gri)       # policy evaluation: the linear system
    dd = np.max(np.abs(vnew - v_pfi))
    v_pfi = vnew
    iterPF += 1
    distPF.append(dd)
time_pfi = time.perf_counter() - timer
print(f"VFI:    {iterVF:4d} iterations, {time_vfi:6.2f} s, policy error {np.max(np.abs(kprime - kprime_true)):.2e}, value error {np.max(np.abs(v - v_true)):.2e}")
print(f"Howard: {iterPF:4d} iterations, {time_pfi:6.2f} s, policy error {np.max(np.abs(kprime_pfi - kprime_true)):.2e}, value error {np.max(np.abs(v_pfi - v_true)):.2e}")

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.semilogy(range(1, iterVF + 1), distVF[1:], label="VFI")
plt.semilogy(range(1, iterPF + 1), np.maximum(distPF[1:], 1e-16), "o-", label="Howard")   # a distance of exactly zero has no logarithm
plt.axhline(mpar.crit, color="gray", linestyle=":", label="tolerance")
plt.title("Distance between iterates")
plt.xlabel("iteration n")
plt.legend()
plt.tight_layout()
plt.show()

**What you should find.** Howard's algorithm needs 5 iterations where VFI needs 337. The policy is
the same. The value is **more** accurate, because the linear system removes the stopping error
altogether: what is left is the interpolation error alone. The distance no longer falls at the
rate $\beta$, it collapses, which is the behaviour of a Newton method. Next week's session takes
this up under the name policy function iteration.

### 6. At home: experiments

`solve_off_grid` wraps parts 4 and 5 in one function. It uses Howard's algorithm because it is
fast. Change one field of `mpar` at a time and compare.

| Try | Question |
| --- | --- |
| `interp` linear against spline, at $N = 50$ | how much does linear interpolation gain over the grid search? Why so little? |
| `nk` 10, 20, 30, 50 with the spline | how few grid points are enough to beat the grid search with $N = 2000$ (policy error $10^{-4}$)? |
| `spacing` log with the spline | does log spacing help now? |

In [ ]:
def solve_off_grid(par, mpar):
    gri = make_grid(par, mpar)
    v, d, n = np.zeros(mpar.nk), np.inf, 0
    while d > mpar.crit and n < mpar.maxiter:
        _, kprime = vfi_update(v, par, mpar, gri)
        vnew = howard_step(kprime, par, mpar, gri)
        d, v, n = np.max(np.abs(vnew - v)), vnew, n + 1
    v_true, kprime_true = closed_form(par, gri.k)
    return n, np.max(np.abs(kprime - kprime_true)), np.max(np.abs(v - v_true))


experiments = {
    "linear, N = 50": NumericalParameters(interp="linear"),
    "linear, N = 200": NumericalParameters(interp="linear", nk=200),
    "spline, N = 10": NumericalParameters(nk=10),
    "spline, N = 20": NumericalParameters(nk=20),
    "spline, N = 30": NumericalParameters(nk=30),
    "spline, N = 50": NumericalParameters(),
    "spline, N = 50, log": NumericalParameters(spacing="log"),
}
print(f"{'run':22s} {'iterations':>10s} {'policy error':>13s} {'value error':>12s}")
for label, m in experiments.items():
    n, pol_err, val_err = solve_off_grid(par, m)
    print(f"{label:22s} {n:10d} {pol_err:13.1e} {val_err:12.1e}")

Note your numbers here. The answers are in the solution notebook, which is posted after the session.

### In practice

The spline came from a package today, linear interpolation and the maximization were written by
hand. In your own work use a tested package for the maximization as well:
`scipy.optimize.minimize_scalar(..., bounds=(a, b), method="bounded")`. It minimizes, so hand it
the **negative** of the right hand side. The check below runs it on the test function of part 3.

In [ ]:
from scipy.optimize import minimize_scalar

res = minimize_scalar(lambda x: -(np.log(x) + np.log(1 - x)), bounds=(0.0, 1.0), method="bounded")   # the negative, because it minimizes
print(f"scipy's bounded minimizer on the test function: x = {res.x:.6f}")